# 01 — Municipal election results 2011–2021

**Owner:** A
**Goal:** one row per municipality per election with registered voters, votes cast, spoilt votes and turnout,
keyed on `muni_code`, with the 2016 boundary changes flagged.

**Reads:** `INTERIM / "results_municipal_<year>_clean.csv"` for 2011, 2016, 2021
(made by `01_clean_municipal_2011/2016/2021.ipynb`)
**Writes:** `INTERIM / "results_lge_2011_2021.csv"`

Run the two setup cells below first, every time.

## Setup

In [ ]:
import sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/Thato20-M/democratic-funnel.git"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = Path("/content/democratic-funnel")
    if REPO_DIR.exists():
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "-q"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", REPO_URL, str(REPO_DIR)], check=True)
else:
    REPO_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents]
                    if (p / "src" / "config.py").exists())

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print("Repo:", REPO_DIR)

In [ ]:
from src import config
from src.config import RAW, INTERIM, PROCESSED, MODELS, FIGURES, SEED
from src.io_utils import read_any, clean_columns, inventory
from src.privacy import drop_personal

import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import warnings; warnings.filterwarnings("ignore", category=FutureWarning)

config.make_dirs(); config.set_seed()
pd.set_option("display.max_columns", 60); pd.set_option("display.width", 160)
sns.set_theme(style="whitegrid"); plt.rcParams["figure.dpi"] = 110
print(config.describe())

## 1. Load the three cleaned files

The year is taken from the file name, not from a column, so older versions of the clean files
(which had no `year` column, or a different column order) still stack correctly.

In [ ]:
YEARS = [2011, 2016, 2021]

frames = []
for year in YEARS:
    path = INTERIM / f"results_municipal_{year}_clean.csv"
    df = clean_columns(pd.read_csv(path)).drop(columns="year", errors="ignore")
    frames.append(df.assign(election=year))
    print(f"{year}: {len(df)} municipalities  <- {path.name}")

raw = pd.concat(frames, ignore_index=True)
raw.head()

## 2. Municipality code and standard column names

`Municipality` looks like `EC101 - Camdeboo [Graaff-Reinet]`: the code comes before the dash, and 2011 adds
the main town in brackets. Metros have letter-only codes (`BUF`, `TSH`, ...).

Turnout is stored as a fraction (0–1), as in the rest of the pipeline.

In [ ]:
parts = raw["municipality"].str.extract(r"^\s*([A-Z]{2,3}\d{0,3})\s*-\s*(.+?)\s*$")
assert parts[0].notna().all(), raw.loc[parts[0].isna(), "municipality"].tolist()

results = pd.DataFrame({
    "election":    raw["election"],
    "province":    raw["province"],
    "muni_code":   parts[0],
    "muni_name":   parts[1].str.replace(r"\s*\[.*\]$", "", regex=True),
    "registered":  raw["registeredvoters"],
    "valid_votes": raw["totalvalidvotes"],
    "spoilt":      raw["spoiltvotes"],
})
results["muni_type"]  = np.where(results["muni_code"].str.contains(r"\d"), "local", "metro")
results["votes_cast"] = results["valid_votes"] + results["spoilt"]
results["turnout"]    = results["votes_cast"] / results["registered"]
results.head()

## 3. Flag the 2016 boundary changes

Before the 2016 election many municipalities were merged or re-drawn (234 → 213 local + metro).
A code can survive a merger while covering a bigger area (e.g. `EC101` Camdeboo absorbed Ikwezi and
Baviaans and almost doubled in voters), so the same code does **not** always mean the same area.

`boundary_2016` per municipality:

| value | meaning |
|---|---|
| `abolished` | code exists in 2011 only (merged into another municipality) |
| `new` | code exists from 2016 only (created by a merger) |
| `resized` | code in both, but registered voters changed by more than normal growth (ratio 2016/2011 outside 0.85–1.5) |
| `none` | no change detected |

Nationally, registered voters grew ~11% from 2011 to 2016, so ratios outside 0.85–1.5 point to boundary changes,
not registration. This is detected from the data: smaller re-draws inside that range (e.g. part of an abolished
municipality going to a neighbour) are not caught, so check against the Municipal Demarcation Board list in
`data/raw/boundaries` when it is available. 2016 → 2021 had no code changes (checked below).

In [ ]:
wide = results.pivot(index="muni_code", columns="election", values="registered")
ratio = wide[2016] / wide[2011]

boundary = pd.Series("none", index=wide.index, name="boundary_2016")
boundary[wide[2011].notna() & wide[2016].isna()] = "abolished"
boundary[wide[2011].isna() & wide[2016].notna()] = "new"
boundary[(ratio < 0.85) | (ratio > 1.5)] = "resized"

results = results.merge(boundary, left_on="muni_code", right_index=True, how="left")
results["comparable_2011"] = results["boundary_2016"].eq("none")

print(boundary.value_counts(), "\n")
print("Resized:")
print(pd.DataFrame({"registered_2011": wide[2011], "registered_2016": wide[2016], "ratio": ratio.round(2)})
      [boundary.eq("resized")].sort_values("ratio"))

In [ ]:
# 2016 -> 2021: same set of codes, and no big jumps in registered voters
codes_16 = set(results.query("election == 2016")["muni_code"])
codes_21 = set(results.query("election == 2021")["muni_code"])
print("Codes that differ 2016 vs 2021:", sorted(codes_16 ^ codes_21) or "none")
print((wide[2021] / wide[2016]).describe().round(3))

## 4. Checks before saving

- 234 municipalities in 2011, 213 in 2016 and 2021 (district municipalities are not on the ward ballot, so they are not rows)
- one row per municipality per election, no missing values
- turnout between 0.2 and 0.9
- national turnout close to the IEC figures: 57.6% (2011), 58.0% (2016), 45.9% (2021)

In [ ]:
counts = results.groupby("election")["muni_code"].nunique()
print(counts, "\n")
assert counts.to_dict() == {2011: 234, 2016: 213, 2021: 213}
assert not results.duplicated(["election", "muni_code"]).any()
assert results.notna().all().all()
assert results["turnout"].between(0.2, 0.9).all()

national = results.groupby("election")[["votes_cast", "registered"]].sum()
national["turnout"] = (national["votes_cast"] / national["registered"]).round(4)
national

In [ ]:
from matplotlib.ticker import PercentFormatter

fig, ax = plt.subplots(figsize=(7, 3.5))
sns.boxplot(data=results, x="election", y="turnout", ax=ax, color="lightsteelblue")
ax.set(title="Municipal turnout by election (ward ballot)", xlabel="", ylabel="Turnout")
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
plt.tight_layout()
plt.show()

## 5. Save output

In [ ]:
out = results[["election", "province", "muni_code", "muni_name", "muni_type",
               "registered", "valid_votes", "spoilt", "votes_cast", "turnout",
               "boundary_2016", "comparable_2011"]].sort_values(["election", "muni_code"])

out_path = INTERIM / "results_lge_2011_2021.csv"
out.to_csv(out_path, index=False)
print(f"Saved {out.shape} to: {out_path}")
out.head()